<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Manh2307/PDF-Rag-Assistant/blob/feature-rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cài ollama

In [1]:
import os, subprocess, time
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
os.environ['OLLAMA_FLASH_ATTENTION'] = 'false'
subprocess.Popen(["ollama", "serve"]); time.sleep(10)

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu j

In [2]:
!ollama pull vicuna:7b-v1.5-q5_1
!ollama pull bge-m3

# Install & import

In [3]:
!pip install -q pypdf chromadb ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [4]:
from pypdf import PdfReader
import ollama
import chromadb

# Download file

In [5]:
# https://drive.google.com/file/d/1lWuq0COKnU9mCfMvTEq54DBLgAh3yYDx/view?usp=drive_link
!gdown 1lWuq0COKnU9mCfMvTEq54DBLgAh3yYDx

Downloading...
From: https://drive.google.com/uc?id=1lWuq0COKnU9mCfMvTEq54DBLgAh3yYDx
To: /content/YOLOv10_Tutorials.pdf
100% 16.6M/16.6M [00:00<00:00, 24.2MB/s]


# 1. Read PDF

In [6]:
pdfPath = "YOLOv10_Tutorials.pdf"
reader = PdfReader(pdfPath)

# số trang
num_pages = len(reader.pages)

# full text extract
full_text = '\n'.join(page.extract_text() or "" for page in reader.pages)

print("Số trang: ", num_pages)
print("Tổng ký tự: ", len(full_text))



Số trang:  20
Tổng ký tự:  23157


# 2. Chunking

In [7]:
def fixed_size_chunk(text, size = 1000, overlap=200):
  # Tách text
  paras = [p.strip() for p in text.split("\n") if p.strip()]

  chunks, current_text = [], ""

  # chunking
  for p in paras:
    if len(current_text) + len(p) + 1 <= size:
      current_text += p + '\n'
    else:
      if current_text:
        chunks.append(current_text.strip())
      current_text = (current_text[-overlap:] + p + '\n') if overlap else (p + '\n')

  if current_text.strip():
    chunks.append(current_text.strip())
  return chunks

chunks = fixed_size_chunk(full_text)

print("Số chunk: ", len(chunks))
char_num = 200
print(f"Đoạn chunk text của {char_num} kí tự:\n ", chunks[0][:char_num])

Số chunk:  30
Đoạn chunk text của 200 kí tự:
  AI VIET NAM – AI COURSE 2024
Tutorial: Phát hiện đối tượng trong ảnh với
YOLOv10
Dinh-Thang Duong, Nguyen-Thuan Duong, Minh-Duc Bui và
Quang-Vinh Dinh
Ngày 20 tháng 6 năm 2024
I. Giới thiệu
Object Det


# 3. Embbeding


In [8]:
def embed(text):
  response = ollama.embed(model="bge-m3", input=text)
  # print(response["embeddings"])
  return response["embeddings"]

# tạo vector database
client = chromadb.Client()
collection = client.get_or_create_collection("rag-db")

collection.add(
    ids = [str(i) for i in range(len(chunks))],
    embeddings = embed(chunks),
    documents = chunks
)

print(f"Đã index {collection.count()} chunks")

Đã index 30 chunks


In [12]:
!ollama list

NAME                   ID              SIZE      MODIFIED      
bge-m3:latest          790764642607    1.2 GB    5 minutes ago    
vicuna:7b-v1.5-q5_1    fa2d15c41d43    5.1 GB    5 minutes ago    


#4. Retrieve


In [9]:
def retrieve(query, k = 4):
  try:
    results = collection.query(
        query_embeddings = embed([query]),
        n_results = k # ask for k most similar res
    )
    # print("\nQuery Results (Text Query):")
    # print(results)
    return results["documents"][0]
  except Exception as e:
    print(f"\nError querying collection: {e}")

QUERY = "YOLOv10 dùng để làm gì?"
for doc in retrieve(QUERY):
  print(doc[:200])
  print("-" * 40)

nh Hoa (Tsinghua University)
đã đề xuất mô hìnhYOLOv10 trong bài báoYOLOv10: Real-Time End-to-End Object
Detection [10]. Với những cải tiến mới, mô hình đã đạt được hiệu suất vượt trội hơn so với các

----------------------------------------
ư mục giải nén, có thể thấy bộ dữ liệu này đã được gán nhãn và đưa vào format
cấu trúc dữ liệu training theo yêu cầu của YOLO. Vì vậy, chúng ta sẽ không cần thực hiện
bước chuẩn bị dữ liệu ở bài này.

----------------------------------------
kết hợp với phần tách còn lại bởi
phép point-wise convolution.
Hình 14: Minh họa Partial self-attention (PSA). Ảnh: [10].
12
AI VIETNAM (AIO2024) aivietnam.edu.vn
IV. Cài đặt chương trình và đánh giá

----------------------------------------
.VI. YOLOv5
YOLOv5 [5], không phải do tác giả gốc phát triển, nhưng được cộng đồng sử dụng rộng rãi từ
năm 2020.
- Điểm mới: Tập trung vào tối ưu hóa và dễ dàng sử dụng với các framework như PyTorch.

----------------------------------------


# RAG

In [11]:
# 1. Tạo mẫu Prompt hướng dẫn LLM cách trả lời
PROMPT = """Bạn là trợ lý hỏi đáp. Dùng các đoạn ngữ cảnh dưới đây để trả lời câu hỏi.
Nếu ngữ cảnh không có thông tin, hãy nói là bạn không biết, đừng bịa.
Trả lời ngắn gọn, chính xác, bằng tiếng Việt.

Ngữ cảnh:
{context}

Câu hỏi: {question}
Trả lời:"""

def rag(question, k=4):
    context = "\n\n".join(retrieve(question, k))

    resp = ollama.chat(
        model="vicuna:7b-v1.5-q5_1",
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=question)}],
        options={"temperature": 0},
    )

    return resp["message"]["content"]

print("Đang xử lý câu trả lời...\n")
print("Câu trả lời từ AI:")
print("-" * 40)
print(rag("YOLOv10 dùng để làm gì?"))

Đang xử lý câu trả lời...

Câu trả lời từ AI:
----------------------------------------
YOLOv10 là một phiên bản của mô hình máy học được sử dụng cho việc phát hiện đối tượng trong hình ảnh. Nó đã đạt được nhiều cải tiến so với các phiên bản YOLO trước đó, với hiệu suất tốt hơn về khía cạnh Độ trễ (Latency) và Số lượng tham số mô hình (Number of parameters) trong khi vẫn giữ được độ chính xác (COCO AP) cao. YOLOv10 có thể được sử dụng cho các ứng dụng như kiểm tra môi trường, giám sát an ninh và quản lý tài nguyên.
